# 03 Raster Preprocessing

## Precipitation Downscaling — Khulna

Objectives:

- Select a reference grid
- Reproject raster datasets
- Resample raster datasets
- Clip rasters to the Khulna study area
- Align all rasters to the same grid
- Export processed rasters
- Verify raster consistency

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
import yaml

from rasterio.mask import mask
from rasterio.warp import reproject, Resampling

In [3]:
PROJECT_ROOT = Path(
    r"E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna"
)

with open(
    PROJECT_ROOT / "environment.yml",
    "r",
    encoding="utf-8"
) as file:
    config = yaml.safe_load(file)

print("Configuration loaded successfully.")
print("Project root:", PROJECT_ROOT)

Configuration loaded successfully.
Project root: E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna


In [4]:
RAW_DATA = PROJECT_ROOT / config["paths"]["raw_data"]
PROCESSED_DATA = PROJECT_ROOT / config["paths"]["processed_data"]

CCS = PROJECT_ROOT / config["paths"]["ccs"]
CDR = PROJECT_ROOT / config["paths"]["cdr"]
CHIRPS = PROJECT_ROOT / config["paths"]["chirps"]
ERA5 = PROJECT_ROOT / config["paths"]["era5"]

GSMAP = PROJECT_ROOT / config["paths"]["gsmap"]
GSMAP_MVK = PROJECT_ROOT / config["paths"]["gsmap_mvk"]
IMERG = PROJECT_ROOT / config["paths"]["imerg"]

LST = PROJECT_ROOT / config["paths"]["lst"]
NDVI = PROJECT_ROOT / config["paths"]["ndvi"]
PDIR = PROJECT_ROOT / config["paths"]["pdir"]
PERSIANN = PROJECT_ROOT / config["paths"]["persiann"]

LAND_VARIABLE = PROJECT_ROOT / config["paths"]["land_variable"]
DISTANCE_SEA = PROJECT_ROOT / config["paths"]["distance_sea"]
BOUNDARY = PROJECT_ROOT / config["paths"]["boundary"]

RASTER_OUTPUT = PROCESSED_DATA / "rasters"
RASTER_OUTPUT.mkdir(parents=True, exist_ok=True)

print("Raster output:", RASTER_OUTPUT)

Raster output: E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\rasters


In [5]:
boundary_files = sorted(BOUNDARY.glob("*.shp"))

if not boundary_files:
    raise FileNotFoundError(
        f"No shapefile found in {BOUNDARY}"
    )

boundary_file = boundary_files[0]

khulna_boundary = gpd.read_file(boundary_file)

print("Boundary file:", boundary_file.name)
print("Boundary CRS:", khulna_boundary.crs)
print("Features:", len(khulna_boundary))

khulna_boundary.head()

Boundary file: Khulna.shp
Boundary CRS: EPSG:4326
Features: 1


,adm2_name,adm2_name1,adm2_name2,adm2_name3,adm2_pcode,adm1_name,adm1_name1,adm1_name2,adm1_name3,adm1_pcode,...,area_sqkm,version,lang,lang1,lang2,lang3,adm2_ref_n,center_lat,center_lon,geometry
0,Khulna,None,None,None,BD4047,Khulna,None,None,None,BD40,...,4457.26501,v03,en,None,None,None,None,22.365739,89.452816,"POLYGON ((89.45595 23.01135, 89.45661 23.01113..."


In [6]:
khulna_boundary = khulna_boundary.dissolve()

print("Dissolved features:", len(khulna_boundary))

Dissolved features: 1


In [7]:
khulna_boundary

,geometry,adm2_name,adm2_name1,adm2_name2,adm2_name3,adm2_pcode,adm1_name,adm1_name1,adm1_name2,adm1_name3,...,valid_to,area_sqkm,version,lang,lang1,lang2,lang3,adm2_ref_n,center_lat,center_lon
0,"POLYGON ((89.45595 23.01135, 89.45661 23.01113...",Khulna,None,None,None,BD4047,Khulna,None,None,None,...,NaT,4457.26501,v03,en,None,None,None,None,22.365739,89.452816


In [8]:
boundary_files = sorted(BOUNDARY.glob("*.shp"))

if not boundary_files:
    raise FileNotFoundError("No boundary shapefile found.")

khulna_boundary = gpd.read_file(boundary_files[0])

print("Boundary CRS:", khulna_boundary.crs)
print("Features:", len(khulna_boundary))

Boundary CRS: EPSG:4326
Features: 1


In [9]:
from rasterio.mask import mask


def clip_raw_raster(
    input_raster,
    output_raster,
    boundary_gdf,
):
    output_raster.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with rasterio.open(input_raster) as src:

        boundary_in_raster_crs = boundary_gdf.to_crs(
            src.crs
        )

        geometries = [
            geometry
            for geometry in boundary_in_raster_crs.geometry
            if geometry is not None
        ]

        clipped_data, clipped_transform = mask(
            src,
            geometries,
            crop=True,
            filled=True,
            nodata=src.nodata,
        )

        clipped_profile = src.profile.copy()

        clipped_profile.update(
            {
                "height": clipped_data.shape[1],
                "width": clipped_data.shape[2],
                "transform": clipped_transform,
                "compress": "lzw",
            }
        )

        with rasterio.open(
            output_raster,
            "w",
            **clipped_profile
        ) as dst:
            dst.write(clipped_data)

In [10]:
test_input = sorted(CHIRPS.glob("*.tif"))[0]

CLIPPED_TEST_FOLDER = (
    PROCESSED_DATA
    / "test_clip"
)

CLIPPED_TEST_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)

test_clip_output = (
    CLIPPED_TEST_FOLDER
    / test_input.name
)

clip_raw_raster(
    input_raster=test_input,
    output_raster=test_clip_output,
    boundary_gdf=khulna_boundary,
)

print("Saved:", test_clip_output)

Saved: E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\test_clip\2017_01.tif


In [11]:
with rasterio.open(test_input) as src:
    print("RAW RASTER")
    print("Width:", src.width)
    print("Height:", src.height)
    print("Bounds:", src.bounds)
    print("Resolution:", src.res)

with rasterio.open(test_clip_output) as src:
    print("\nCLIPPED RASTER")
    print("Width:", src.width)
    print("Height:", src.height)
    print("Bounds:", src.bounds)
    print("Resolution:", src.res)

RAW RASTER
Width: 12
Height: 28
Bounds: BoundingBox(left=89.20040802594113, bottom=21.65009903320208, right=89.80041077051024, top=23.05010543719667)
Resolution: (0.050000228714092564, 0.050000228714092564)

CLIPPED RASTER
Width: 12
Height: 28
Bounds: BoundingBox(left=89.20040802594113, bottom=21.65009903320208, right=89.80041077051024, top=23.05010543719667)
Resolution: (0.050000228714092564, 0.050000228714092564)


In [12]:
import numpy as np
import rasterio

with rasterio.open(test_input) as src:
    raw = src.read(1, masked=True)

with rasterio.open(test_clip_output) as src:
    clipped = src.read(1, masked=True)

print("Raw valid pixels     :", raw.count())
print("Clipped valid pixels :", clipped.count())
print("Raw masked pixels    :", np.ma.count_masked(raw))
print("Clipped masked pixels:", np.ma.count_masked(clipped))

Raw valid pixels     : 336
Clipped valid pixels : 336
Raw masked pixels    : 0
Clipped masked pixels: 0


In [13]:
datasets = {
    "CHIRPS": CHIRPS,
    "ERA5": ERA5,
    "CCS": CCS,
    "CDR": CDR,
    "IMERG": IMERG,
    "GSMaP_Gauge": GSMAP,
    "GSMaP_MVK": GSMAP_MVK,
    "NDVI": NDVI,
    "LST": LST,
    "PDIR": PDIR,
    "PERSIANN": PERSIANN,
}

for name, folder in datasets.items():

    tif = sorted(folder.glob("*.tif"))[0]

    with rasterio.open(tif) as src:

        print("="*60)
        print(name)
        print("CRS       :", src.crs)
        print("Width     :", src.width)
        print("Height    :", src.height)
        print("Resolution:", src.res)
        print("Bounds    :", src.bounds)

CHIRPS
CRS       : GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
Width     : 12
Height    : 28
Resolution: (0.050000228714092564, 0.050000228714092564)
Bounds    : BoundingBox(left=89.20040802594113, bottom=21.65009903320208, right=89.80041077051024, top=23.05010543719667)
ERA5
CRS       : GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
Width     : 4
Height    : 7
Resolution: (0.2500011435704628, 0.2500011435704628)
Bounds    : BoundingBox(left=89.00040711108477, bottom=21.5000983470598, right=90.00041168536661, top=23.25010635205304)
CCS
CRS       : GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PR

In [14]:
REFERENCE_RASTER = sorted(NDVI.glob("*.tif"))[0]

with rasterio.open(REFERENCE_RASTER) as ref:
    print("Reference:", REFERENCE_RASTER.name)
    print("CRS:", ref.crs)
    print("Shape:", ref.height, ref.width)
    print("Resolution:", ref.res)
    print("Bounds:", ref.bounds)

Reference: NDVI_2017_1.tif
CRS: GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
Shape: 151 59
Resolution: (0.008983152841195215, 0.008983152841195215)
Bounds: BoundingBox(left=89.22965717159208, bottom=21.658381500121664, right=89.7596631892226, top=23.014837579142142)


In [25]:
# ==========================================================
# 03_Raster_Preprocessing_FIXED.py
# Full Raster Processing Code
# Fixes CDR and PDIR CRS/metadata problems
# ==========================================================

from pathlib import Path
import warnings
import traceback

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import yaml

from rasterio.warp import reproject, Resampling
from rasterio.features import geometry_mask
from rasterio.transform import Affine


# ==========================================================
# 1. Project Root and Configuration
# ==========================================================

PROJECT_ROOT = Path(
    r"E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna"
)

CONFIG_FILE = PROJECT_ROOT / "environment.yml"

if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Configuration file not found: {CONFIG_FILE}")

with open(CONFIG_FILE, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

if not isinstance(config, dict) or "paths" not in config:
    raise ValueError(
        "environment.yml must contain a top-level 'paths' section."
    )

print("Configuration loaded successfully.")
print("Project root:", PROJECT_ROOT)


# ==========================================================
# 2. Define Paths
# ==========================================================

def project_path(config_key):
    """Return an absolute project path from environment.yml."""
    if config_key not in config["paths"]:
        raise KeyError(
            f"Missing paths.{config_key} in environment.yml"
        )

    configured_path = Path(config["paths"][config_key])

    if configured_path.is_absolute():
        return configured_path

    return PROJECT_ROOT / configured_path


PROCESSED_DATA = project_path("processed_data")

CCS = project_path("ccs")
CDR = project_path("cdr")
CHIRPS = project_path("chirps")
ERA5 = project_path("era5")

GSMAP = project_path("gsmap")
GSMAP_MVK = project_path("gsmap_mvk")
IMERG = project_path("imerg")

LST = project_path("lst")
NDVI = project_path("ndvi")

PDIR = project_path("pdir")
PERSIANN = project_path("persiann")

LAND_VARIABLE = project_path("land_variable")
DISTANCE_SEA = project_path("distance_sea")
BOUNDARY = project_path("boundary")

ALIGNED_OUTPUT = PROCESSED_DATA / "rasters_aligned"
LOG_OUTPUT = PROCESSED_DATA / "logs"

ALIGNED_OUTPUT.mkdir(parents=True, exist_ok=True)
LOG_OUTPUT.mkdir(parents=True, exist_ok=True)

print("Aligned output:", ALIGNED_OUTPUT)
print("Log output:", LOG_OUTPUT)


# ==========================================================
# 3. Processing Settings
# ==========================================================

# CDR and PDIR CRS overrides are defined after the reference raster
# has been opened. This avoids CRS.from_epsg(), which can fail when
# multiple incompatible PROJ installations are present.

# Set True to overwrite existing processed files.
OVERWRITE_EXISTING = True

# A standard finite output NoData value is safer than NaN for GeoTIFF.
OUTPUT_NODATA = -9999.0


# ==========================================================
# 4. Utility Functions
# ==========================================================

def list_tiff_files(folder):
    """Return all TIFF files from a folder."""
    if not folder.exists():
        warnings.warn(f"Input folder not found: {folder}")
        return []

    files = list(folder.glob("*.tif")) + list(folder.glob("*.tiff"))
    return sorted(set(files))


def validate_transform(transform, raster_path):
    """Check whether a raster transform is usable."""
    if transform is None:
        raise ValueError(f"Missing transform: {raster_path}")

    if not isinstance(transform, Affine):
        transform = Affine(*transform[:6])

    values = np.asarray(
        [
            transform.a,
            transform.b,
            transform.c,
            transform.d,
            transform.e,
            transform.f,
        ],
        dtype="float64",
    )

    if not np.all(np.isfinite(values)):
        raise ValueError(f"Invalid transform values: {raster_path}")

    if transform.a == 0 or transform.e == 0:
        raise ValueError(f"Zero pixel size in transform: {raster_path}")

    return transform


def normalize_source_nodata(src, source_data):
    """
    Return a safe source NoData value.

    Some rasters have an invalid NoData value outside their data type
    range. In that case, use None and mask non-finite floating values
    directly in the source array.
    """
    src_nodata = src.nodata

    if src_nodata is None:
        return None

    try:
        src_nodata_float = float(src_nodata)
    except (TypeError, ValueError):
        return None

    if not np.isfinite(src_nodata_float):
        return None

    dtype = np.dtype(source_data.dtype)

    if np.issubdtype(dtype, np.integer):
        limits = np.iinfo(dtype)
        if not limits.min <= src_nodata_float <= limits.max:
            return None

    return src_nodata_float


def select_resampling(dataset_name, file_name=""):
    """
    Use nearest-neighbour for categorical rasters.
    Use bilinear interpolation for continuous rasters.
    """
    text = f"{dataset_name} {file_name}".lower()

    categorical_keywords = [
        "landcover",
        "land_cover",
        "lulc",
        "landuse",
        "land_use",
        "class",
        "category",
        "soil_type",
    ]

    if any(keyword in text for keyword in categorical_keywords):
        return Resampling.nearest

    return Resampling.bilinear


# ==========================================================
# 5. Load Khulna Boundary
# ==========================================================

boundary_files = sorted(BOUNDARY.glob("*.shp"))

if not boundary_files:
    raise FileNotFoundError(
        f"No boundary shapefile found in: {BOUNDARY}"
    )

boundary_file = boundary_files[0]
khulna_boundary = gpd.read_file(boundary_file)

if khulna_boundary.crs is None:
    raise ValueError("Khulna boundary does not have a CRS.")

khulna_boundary = khulna_boundary[
    khulna_boundary.geometry.notna()
].copy()

khulna_boundary = khulna_boundary[
    ~khulna_boundary.geometry.is_empty
].copy()

if khulna_boundary.empty:
    raise ValueError("Khulna boundary contains no valid geometry.")

# Repair invalid geometry where possible.
khulna_boundary["geometry"] = khulna_boundary.geometry.buffer(0)
khulna_boundary = khulna_boundary.dissolve()

print("Boundary file:", boundary_file.name)
print("Boundary CRS:", khulna_boundary.crs)


# ==========================================================
# 6. Select Reference Raster
# ==========================================================

ndvi_files = list_tiff_files(NDVI)

if not ndvi_files:
    raise FileNotFoundError(
        f"No NDVI TIFF file found in: {NDVI}"
    )

REFERENCE_RASTER = ndvi_files[0]

with rasterio.open(REFERENCE_RASTER) as ref:
    reference_crs = ref.crs
    reference_transform = validate_transform(
        ref.transform,
        REFERENCE_RASTER,
    )
    reference_width = ref.width
    reference_height = ref.height
    reference_bounds = ref.bounds
    reference_resolution = ref.res
    reference_profile = ref.profile.copy()

if reference_crs is None:
    raise ValueError("Reference raster does not have a CRS.")

# Reuse the CRS already embedded in the valid NDVI reference raster.
# Do not call CRS.from_epsg(4326), because the current Conda environment
# has a conflicting/outdated PROJ database.
SOURCE_CRS_OVERRIDES = {
    "CDR": reference_crs,
    "PDIR": reference_crs,
}

print("\nReference raster information")
print("-" * 60)
print("File:", REFERENCE_RASTER.name)
print("CRS:", reference_crs)
print("Width:", reference_width)
print("Height:", reference_height)
print("Resolution:", reference_resolution)
print("Bounds:", reference_bounds)


# Prepare the boundary mask only once.
boundary_reference_crs = khulna_boundary.to_crs(reference_crs)

reference_geometries = [
    geometry
    for geometry in boundary_reference_crs.geometry
    if geometry is not None and not geometry.is_empty
]

if not reference_geometries:
    raise ValueError("No valid boundary geometry found.")

inside_boundary_mask = geometry_mask(
    reference_geometries,
    out_shape=(reference_height, reference_width),
    transform=reference_transform,
    invert=True,
)


# ==========================================================
# 7. Dataset Definitions
# ==========================================================

monthly_datasets = {
    "CCS": CCS,
    "CDR": CDR,
    "CHIRPS": CHIRPS,
    "ERA5": ERA5,
    "GSMaP_Gauge": GSMAP,
    "GSMaP_MVK": GSMAP_MVK,
    "IMERG": IMERG,
    "LST": LST,
    "NDVI": NDVI,
    "PDIR": PDIR,
    "PERSIANN": PERSIANN,
}

static_datasets = {}

distance_meter = DISTANCE_SEA / "Distance_Sea_meter.tif"

if distance_meter.exists():
    static_datasets["Distance_Sea_meter"] = distance_meter
else:
    print("Warning: Distance_Sea_meter.tif was not found.")

for raster_file in list_tiff_files(LAND_VARIABLE):
    static_datasets[f"Land_{raster_file.stem}"] = raster_file

print("\nMonthly datasets:", len(monthly_datasets))
print("Static rasters:", len(static_datasets))


# ==========================================================
# 8. Raster Processing Function
# ==========================================================

def process_raster(
    input_raster,
    output_raster,
    dataset_name,
    resampling_method=Resampling.bilinear,
):
    """
    Reproject and resample an input raster to the NDVI reference grid,
    apply the Khulna boundary mask, and save a Float32 GeoTIFF.

    CDR and PDIR are forced to EPSG:4326 to bypass incorrect CRS
    metadata in the source files.
    """
    output_raster.parent.mkdir(parents=True, exist_ok=True)

    if output_raster.exists() and not OVERWRITE_EXISTING:
        return {
            "Source_CRS": "Existing output",
            "Used_CRS": str(reference_crs),
            "Source_NoData": None,
            "Valid_Output_Pixels": None,
            "Skipped": True,
        }

    with rasterio.open(input_raster) as src:
        if src.count < 1:
            raise ValueError(f"No raster band found: {input_raster}")

        source_transform = validate_transform(
            src.transform,
            input_raster,
        )

        original_source_crs = src.crs
        source_crs = SOURCE_CRS_OVERRIDES.get(
            dataset_name,
            original_source_crs,
        )

        if source_crs is None:
            raise ValueError(
                f"Missing CRS and no override configured: {input_raster}"
            )

        source_data = src.read(1)

        if source_data.size == 0:
            raise ValueError(f"Empty raster array: {input_raster}")

        src_nodata = normalize_source_nodata(src, source_data)

        # Convert the source to float32 so invalid values can be replaced.
        source_float = source_data.astype(np.float32, copy=False)

        # Replace non-finite source pixels with the output NoData value.
        source_float = np.where(
            np.isfinite(source_float),
            source_float,
            OUTPUT_NODATA,
        ).astype(np.float32)

        # When a valid source NoData exists, retain it for reprojection.
        if src_nodata is not None:
            source_for_reproject = source_float
            effective_src_nodata = float(src_nodata)
        else:
            source_for_reproject = source_float
            effective_src_nodata = OUTPUT_NODATA

        destination = np.full(
            (reference_height, reference_width),
            OUTPUT_NODATA,
            dtype=np.float32,
        )

        reproject(
            source=source_for_reproject,
            destination=destination,
            src_transform=source_transform,
            src_crs=source_crs,
            src_nodata=effective_src_nodata,
            dst_transform=reference_transform,
            dst_crs=reference_crs,
            dst_nodata=OUTPUT_NODATA,
            resampling=resampling_method,
            init_dest_nodata=True,
            num_threads=2,
        )

    # Apply Khulna boundary mask.
    destination[~inside_boundary_mask] = OUTPUT_NODATA

    # Replace any unexpected non-finite values.
    destination[~np.isfinite(destination)] = OUTPUT_NODATA

    valid_output = (
        inside_boundary_mask
        & (destination != OUTPUT_NODATA)
        & np.isfinite(destination)
    )

    output_profile = reference_profile.copy()
    output_profile.update(
        {
            "driver": "GTiff",
            "height": reference_height,
            "width": reference_width,
            "transform": reference_transform,
            "crs": reference_crs,
            "count": 1,
            "dtype": "float32",
            "nodata": OUTPUT_NODATA,
            "compress": "lzw",
            "predictor": 3,
            "tiled": False,
            "BIGTIFF": "IF_SAFER",
        }
    )

    with rasterio.open(output_raster, "w", **output_profile) as dst:
        dst.write(destination, 1)
        dst.set_band_description(1, dataset_name)

    return {
        "Source_CRS": str(original_source_crs),
        "Used_CRS": str(source_crs),
        "Source_NoData": src_nodata,
        "Valid_Output_Pixels": int(valid_output.sum()),
        "Skipped": False,
    }


# ==========================================================
# 9. Process All Monthly Rasters
# ==========================================================

monthly_success = []
monthly_failed = []

for dataset_name, input_folder in monthly_datasets.items():
    raster_files = list_tiff_files(input_folder)

    output_folder = ALIGNED_OUTPUT / "monthly" / dataset_name
    output_folder.mkdir(parents=True, exist_ok=True)

    print(
        f"\nProcessing {dataset_name}: "
        f"{len(raster_files)} file(s)"
    )

    if not raster_files:
        monthly_failed.append(
            {
                "Dataset": dataset_name,
                "Input_File": "",
                "Error_Type": "FileNotFoundError",
                "Error": f"No TIFF files found in {input_folder}",
            }
        )
        print("  No TIFF files found.")
        continue

    for index, input_file in enumerate(raster_files, start=1):
        output_file = output_folder / input_file.name

        resampling_method = select_resampling(
            dataset_name,
            input_file.name,
        )

        try:
            result = process_raster(
                input_raster=input_file,
                output_raster=output_file,
                dataset_name=dataset_name,
                resampling_method=resampling_method,
            )

            monthly_success.append(
                {
                    "Dataset": dataset_name,
                    "Input_File": input_file.name,
                    "Output_File": str(output_file),
                    "Resampling": resampling_method.name,
                    "Original_Source_CRS": result["Source_CRS"],
                    "Used_Source_CRS": result["Used_CRS"],
                    "CRS_Override": dataset_name in SOURCE_CRS_OVERRIDES,
                    "Source_NoData": result["Source_NoData"],
                    "Valid_Output_Pixels": result["Valid_Output_Pixels"],
                    "Skipped": result["Skipped"],
                    "Status": "Success",
                }
            )

            if (
                index == 1
                or index == len(raster_files)
                or index % 12 == 0
            ):
                print(
                    f"  Completed: {index}/{len(raster_files)}"
                )

        except Exception as error:
            monthly_failed.append(
                {
                    "Dataset": dataset_name,
                    "Input_File": input_file.name,
                    "Error_Type": type(error).__name__,
                    "Error": str(error),
                    "Traceback": traceback.format_exc(),
                }
            )

            print(
                f"  FAILED: {dataset_name}/{input_file.name}"
            )
            print(
                f"    {type(error).__name__}: {error}"
            )


# ==========================================================
# 10. Process Static Rasters
# ==========================================================

static_success = []
static_failed = []

static_output_folder = ALIGNED_OUTPUT / "static"
static_output_folder.mkdir(parents=True, exist_ok=True)

print("\nProcessing static rasters")

for variable_name, input_file in static_datasets.items():
    output_file = static_output_folder / f"{variable_name}.tif"

    resampling_method = select_resampling(
        variable_name,
        input_file.name,
    )

    try:
        result = process_raster(
            input_raster=input_file,
            output_raster=output_file,
            dataset_name=variable_name,
            resampling_method=resampling_method,
        )

        static_success.append(
            {
                "Variable": variable_name,
                "Input_File": input_file.name,
                "Output_File": str(output_file),
                "Resampling": resampling_method.name,
                "Original_Source_CRS": result["Source_CRS"],
                "Used_Source_CRS": result["Used_CRS"],
                "Source_NoData": result["Source_NoData"],
                "Valid_Output_Pixels": result["Valid_Output_Pixels"],
                "Skipped": result["Skipped"],
                "Status": "Success",
            }
        )

        print("  Processed:", variable_name)

    except Exception as error:
        static_failed.append(
            {
                "Variable": variable_name,
                "Input_File": input_file.name,
                "Error_Type": type(error).__name__,
                "Error": str(error),
                "Traceback": traceback.format_exc(),
            }
        )

        print(
            f"  FAILED: {variable_name} - "
            f"{type(error).__name__}: {error}"
        )


# ==========================================================
# 11. Create Processing Logs
# ==========================================================

monthly_success_df = pd.DataFrame(monthly_success)
monthly_failed_df = pd.DataFrame(monthly_failed)
static_success_df = pd.DataFrame(static_success)
static_failed_df = pd.DataFrame(static_failed)


# ==========================================================
# 12. Grid Information Function
# ==========================================================

def get_grid_information(raster_path):
    """Return raster grid and valid-pixel information."""
    with rasterio.open(raster_path) as src:
        data = src.read(1, masked=True)

        return {
            "Dataset_Folder": raster_path.parent.name,
            "File": raster_path.name,
            "Full_Path": str(raster_path),
            "CRS": str(src.crs),
            "Width": src.width,
            "Height": src.height,
            "Resolution_X": abs(src.res[0]),
            "Resolution_Y": abs(src.res[1]),
            "Transform": tuple(src.transform),
            "Bounds": tuple(src.bounds),
            "NoData": src.nodata,
            "Valid_Pixels": int(data.count()),
            "Masked_Pixels": int(np.ma.count_masked(data)),
        }


# ==========================================================
# 13. Check All Processed Rasters
# ==========================================================

grid_records = []

processed_raster_files = sorted(ALIGNED_OUTPUT.rglob("*.tif"))

for processed_file in processed_raster_files:
    try:
        grid_records.append(
            get_grid_information(processed_file)
        )
    except Exception as error:
        print(
            "Grid check failed:",
            processed_file.name,
            error,
        )

grid_check_df = pd.DataFrame(grid_records)


# ==========================================================
# 14. Alignment Summary
# ==========================================================

if grid_check_df.empty:
    alignment_summary = pd.Series(
        {
            "Total Files": 0,
            "Unique CRS": 0,
            "Unique Width": 0,
            "Unique Height": 0,
            "Unique Resolution X": 0,
            "Unique Resolution Y": 0,
            "Unique Transform": 0,
            "Unique Bounds": 0,
        }
    )
else:
    alignment_summary = pd.Series(
        {
            "Total Files": len(grid_check_df),
            "Unique CRS": grid_check_df["CRS"].nunique(),
            "Unique Width": grid_check_df["Width"].nunique(),
            "Unique Height": grid_check_df["Height"].nunique(),
            "Unique Resolution X": grid_check_df[
                "Resolution_X"
            ].nunique(),
            "Unique Resolution Y": grid_check_df[
                "Resolution_Y"
            ].nunique(),
            "Unique Transform": grid_check_df[
                "Transform"
            ].nunique(),
            "Unique Bounds": grid_check_df["Bounds"].nunique(),
        }
    )


# ==========================================================
# 15. Empty Raster Check
# ==========================================================

if grid_check_df.empty:
    empty_rasters_df = pd.DataFrame()
else:
    empty_rasters_df = grid_check_df[
        grid_check_df["Valid_Pixels"] == 0
    ].copy()


# ==========================================================
# 16. Dataset Completion Summary
# ==========================================================

expected_monthly_counts = {
    dataset_name: len(list_tiff_files(folder))
    for dataset_name, folder in monthly_datasets.items()
}

completion_records = []

for dataset_name, expected_count in expected_monthly_counts.items():
    success_count = 0
    failed_count = 0
    empty_count = 0

    if not monthly_success_df.empty:
        success_count = int(
            (
                monthly_success_df["Dataset"] == dataset_name
            ).sum()
        )

    if not monthly_failed_df.empty:
        failed_count = int(
            (
                monthly_failed_df["Dataset"] == dataset_name
            ).sum()
        )

    if not empty_rasters_df.empty:
        empty_count = int(
            (
                empty_rasters_df["Dataset_Folder"] == dataset_name
            ).sum()
        )

    completion_records.append(
        {
            "Dataset": dataset_name,
            "Expected_Files": expected_count,
            "Successful_Files": success_count,
            "Failed_Files": failed_count,
            "Empty_Output_Files": empty_count,
            "Complete": (
                expected_count > 0
                and success_count == expected_count
                and failed_count == 0
                and empty_count == 0
            ),
        }
    )

completion_summary_df = pd.DataFrame(completion_records)


# ==========================================================
# 17. Save Logs
# ==========================================================

monthly_success_df.to_csv(
    LOG_OUTPUT / "monthly_raster_processing_log.csv",
    index=False,
)

monthly_failed_df.to_csv(
    LOG_OUTPUT / "monthly_raster_failed_log.csv",
    index=False,
)

static_success_df.to_csv(
    LOG_OUTPUT / "static_raster_processing_log.csv",
    index=False,
)

static_failed_df.to_csv(
    LOG_OUTPUT / "static_raster_failed_log.csv",
    index=False,
)

grid_check_df.to_csv(
    LOG_OUTPUT / "aligned_raster_grid_check.csv",
    index=False,
)

empty_rasters_df.to_csv(
    LOG_OUTPUT / "empty_rasters.csv",
    index=False,
)

completion_summary_df.to_csv(
    LOG_OUTPUT / "dataset_completion_summary.csv",
    index=False,
)


# ==========================================================
# 18. Final Report
# ==========================================================

print("\n" + "=" * 80)
print("RASTER PREPROCESSING COMPLETED")
print("=" * 80)

print("Reference raster      :", REFERENCE_RASTER)
print("Output folder         :", ALIGNED_OUTPUT)
print("Monthly successful    :", len(monthly_success_df))
print("Monthly failed        :", len(monthly_failed_df))
print("Static successful     :", len(static_success_df))
print("Static failed         :", len(static_failed_df))
print("Processed raster files:", len(grid_check_df))
print("Empty rasters         :", len(empty_rasters_df))

print("\nAlignment summary")
print("-" * 80)
print(alignment_summary.to_string())

print("\nDataset completion summary")
print("-" * 80)
if completion_summary_df.empty:
    print("No monthly datasets were found.")
else:
    print(completion_summary_df.to_string(index=False))

print("=" * 80)

if not monthly_failed_df.empty:
    print("\nMonthly failures")
    print("-" * 80)
    print(
        monthly_failed_df[
            ["Dataset", "Input_File", "Error_Type", "Error"]
        ].to_string(index=False)
    )

if not static_failed_df.empty:
    print("\nStatic failures")
    print("-" * 80)
    print(
        static_failed_df[
            ["Variable", "Input_File", "Error_Type", "Error"]
        ].to_string(index=False)
    )

if not empty_rasters_df.empty:
    print("\nEmpty rasters")
    print("-" * 80)
    print(
        empty_rasters_df[
            ["Dataset_Folder", "File", "Valid_Pixels"]
        ].to_string(index=False)
    )

print("\nLog files saved in:", LOG_OUTPUT)
print("=" * 80)

Configuration loaded successfully.
Project root: E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna
Aligned output: E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\rasters_aligned
Log output: E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\logs
Boundary file: Khulna.shp
Boundary CRS: EPSG:4326

Reference raster information
------------------------------------------------------------
File: NDVI_2017_1.tif
CRS: GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
Width: 59
Height: 151
Resolution: (0.008983152841195215, 0.008983152841195215)
Bounds: BoundingBox(left=89.22965717159208, bottom=21.658381500121664, right=89.7596631892226, top=23.014837579142142)

Monthly datasets: 11
Static rasters: 4

Processing CCS: 72 file(s

In [26]:
# ==========================================================
# Raster Alignment Verification
# Check CRS, Width, Height, Resolution, Transform, Bounds
# ==========================================================

from pathlib import Path
import rasterio
import pandas as pd


# আপনার aligned raster folder
ALIGNED_FOLDER = Path(
    r"E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\rasters_aligned"
)


# সব raster খুঁজে বের করা
raster_files = sorted(
    ALIGNED_FOLDER.rglob("*.tif")
)

print("Total raster files:", len(raster_files))


records = []


for raster in raster_files:

    with rasterio.open(raster) as src:

        records.append(
            {
                "Dataset": raster.parent.name,
                "File": raster.name,

                "CRS": str(src.crs),

                "Width": src.width,
                "Height": src.height,

                "Resolution_X": src.res[0],
                "Resolution_Y": src.res[1],

                "Transform": tuple(src.transform),

                "Bounds": tuple(src.bounds),
            }
        )


# DataFrame তৈরি
df = pd.DataFrame(records)


# Preview
display(df.head())


# ==========================================================
# Alignment Summary
# ==========================================================

summary = pd.DataFrame(
    {
        "Parameter": [
            "CRS",
            "Width",
            "Height",
            "Resolution_X",
            "Resolution_Y",
            "Transform",
            "Bounds",
        ],

        "Unique_Count": [
            df["CRS"].nunique(),
            df["Width"].nunique(),
            df["Height"].nunique(),
            df["Resolution_X"].nunique(),
            df["Resolution_Y"].nunique(),
            df["Transform"].nunique(),
            df["Bounds"].nunique(),
        ]
    }
)


print("\n========== ALIGNMENT CHECK ==========")

display(summary)


# ==========================================================
# Final Decision
# ==========================================================

if (
    df["CRS"].nunique() == 1
    and df["Width"].nunique() == 1
    and df["Height"].nunique() == 1
    and df["Resolution_X"].nunique() == 1
    and df["Resolution_Y"].nunique() == 1
    and df["Transform"].nunique() == 1
    and df["Bounds"].nunique() == 1
):

    print(
        "\n✅ ALL RASTERS ARE PERFECTLY ALIGNED"
    )

else:

    print(
        "\n❌ ALIGNMENT PROBLEM FOUND"
    )


# ==========================================================
# কোন raster আলাদা হলে বের করবে
# ==========================================================

for col in [
    "CRS",
    "Width",
    "Height",
    "Resolution_X",
    "Resolution_Y",
    "Transform",
    "Bounds",
]:

    print("\nDifferent values in:", col)

    print(
        df[col]
        .value_counts()
    )

Total raster files: 796


,Dataset,File,CRS,Width,Height,Resolution_X,Resolution_Y,Transform,Bounds
0,CCS,2017_01.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",59,151,0.008983,0.008983,"(0.008983152841195215, 0.0, 89.22965717159208,...","(89.22965717159208, 21.658381500121664, 89.759..."
1,CCS,2017_02.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",59,151,0.008983,0.008983,"(0.008983152841195215, 0.0, 89.22965717159208,...","(89.22965717159208, 21.658381500121664, 89.759..."
2,CCS,2017_03.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",59,151,0.008983,0.008983,"(0.008983152841195215, 0.0, 89.22965717159208,...","(89.22965717159208, 21.658381500121664, 89.759..."
3,CCS,2017_04.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",59,151,0.008983,0.008983,"(0.008983152841195215, 0.0, 89.22965717159208,...","(89.22965717159208, 21.658381500121664, 89.759..."
4,CCS,2017_05.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",59,151,0.008983,0.008983,"(0.008983152841195215, 0.0, 89.22965717159208,...","(89.22965717159208, 21.658381500121664, 89.759..."



========== ALIGNMENT CHECK ==========


,Parameter,Unique_Count
0,CRS,1
1,Width,1
2,Height,1
3,Resolution_X,1
4,Resolution_Y,1
5,Transform,1
6,Bounds,1



✅ ALL RASTERS ARE PERFECTLY ALIGNED

Different values in: CRS
CRS
GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]    796
Name: count, dtype: int64

Different values in: Width
Width
59    796
Name: count, dtype: int64

Different values in: Height
Height
151    796
Name: count, dtype: int64

Different values in: Resolution_X
Resolution_X
0.008983    796
Name: count, dtype: int64

Different values in: Resolution_Y
Resolution_Y
0.008983    796
Name: count, dtype: int64

Different values in: Transform
Transform
(0.008983152841195215, 0.0, 89.22965717159208, 0.0, -0.008983152841195215, 23.014837579142142, 0.0, 0.0, 1.0)    796
Name: count, dtype: int64

Different values in: Bounds
Bounds
(89.22965717159208, 21.658381500121664, 89.7596631892226, 23.014837579142142)    796
Name: count, dtype: int64
